#Setup and Imports

In [ ]:
# ✅ Install core libraries (run this in its own cell)
!pip install transformers datasets accelerate sentence-transformers --quiet

In [ ]:
# 📦 Section 1: Imports
import torch
import re
import csv
import random
import numpy as np

from datasets import load_dataset
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

#Hugging Face Login

In [ ]:
!huggingface-cli login

#Model Embedding and Initialization

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.1"  # Change if needed
tokenizer = AutoTokenizer.from_pretrained(model_id, token=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    token=True
)
llm = pipeline("text-generation", model=model, tokenizer=tokenizer)

# 🔍 Sentence transformer for cosine similarity
embedder = SentenceTransformer("all-MiniLM-L6-v2")




#Prompt Template

In [ ]:
# 🎛️ Section 3.1: Prompt Templates
prompt_templates = {
    "solve": lambda prompt: f"""{prompt}

Answer the question step by step. Conclude with '#### [final answer]'.""",

    "critique": lambda prompt, x0: f"""You are an expert tutor reviewing a student's math answer.

Question:
{prompt}

Student's Answer:
{x0}

Provide helpful feedback to improve the answer. Be specific, constructive, and include any mistakes and suggestions.""",

    "student_feedback": lambda prompt, x0: f"""You are a student trying to self-evaluate your math answer.

Question:
{prompt}

Your Answer:
{x0}

What might be wrong? Give a suggestion to improve your own answer."""
}


#Dispatcher Function

*   Current state: there are two seperate instantiations for a base model and an expert model
*   Target: one model pipeline sinstnace (e.g. Mistrl via HF pipeline) shared across roles

**Implication**:  the only thing that should differ between "base" and "expert" is the template and not the model itself.



In [ ]:
# 🔁 Section 3.2: Unified Dispatcher Function
def run_model(prompt, mode="solve", x0=None, temp=0.3, max_tokens=256):
    if mode == "solve":
        formatted_prompt = prompt_templates["solve"](prompt)
    elif mode == "critique":
        if x0 is None:
            raise ValueError("x0 is required for expert critique mode.")
        formatted_prompt = prompt_templates["critique"](prompt, x0)
    elif mode == "student_feedback":
        if x0 is None:
            raise ValueError("x0 is required for student_feedback mode.")
        formatted_prompt = prompt_templates["student_feedback"](prompt, x0)
    else:
        raise ValueError(f"Unsupported mode: {mode}")

    response = llm(
        formatted_prompt,
        max_new_tokens=max_tokens,
        temperature=temp,
        do_sample=True
    )[0]["generated_text"]

    return response

# Utility Functions




In [ ]:
# 🧰 Section 4: Utility Functions

def extract_final_answer(text):
    match = re.search(r"####\s*(-?\d+)", text)
    return int(match.group(1)) if match else None

def extract_answer(text):
    for token in text.split():
        if token.isdigit():
            return int(token)
    return None

def compute_similarity(fb1, fb2):
    vec1 = embedder.encode([fb1])
    vec2 = embedder.encode([fb2])
    return float(cosine_similarity(vec1, vec2)[0][0])

def save_results_to_csv(result, features, file_path):
    with open(file_path, 'a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        row = [result.get(feature, "") for feature in features]
        writer.writerow(row)


# Experimental Loop

In [ ]:
# 🧪 Section 5: Run Experiment on GSM8K

gsm8k = load_dataset("gsm8k", "main", split="train[:5%]")
results = []

for i, example in enumerate(gsm8k):
    input_prompt = example["question"]
    ground_truth = extract_final_answer(example["answer"])

    if ground_truth is None:
        print(f"⚠️ Skipping due to invalid label:\n{example['answer']}")
        continue

    print(f"\n--- Task {i+1} ---")
    print(f"Prompt:\n{input_prompt}")

    # Step 1: Generate initial solution
    x0 = run_model(input_prompt, mode="solve")
    print("Initial Answer:\n", x0.strip())

    # Step 2: Generate expert feedback
    p_exp = run_model(input_prompt, mode="critique", x0=x0)
    print("Expert Feedback:\n", p_exp.strip())

    # Step 3: Generate student feedback (amateur)
    p_stu = run_model(input_prompt, mode="student_feedback", x0=x0)
    print("Student Feedback:\n", p_stu.strip())

    # Step 4: Compute similarity
    score = compute_similarity(p_stu, p_exp)
    print("Feedback Similarity Score:", round(score, 3))

    # Step 5: Refined output using both feedbacks
    ref_prompt = f"""Refine this student's answer based on expert and self feedback.

Question: {input_prompt}

Original Answer:
{x0}

Expert Feedback:
{p_exp}

Student Feedback:
{p_stu}

Provide a better answer. End with '#### [answer]'."""

    x1 = llm(ref_prompt, max_new_tokens=256, temperature=0.3, do_sample=True)[0]["generated_text"]
    print("Refined Answer:\n", x1.strip())

    # Step 6: Evaluation
    pred = extract_final_answer(x1)
    correct = int(pred == ground_truth) if pred is not None else 0
    print(f"Prediction: {pred} | Ground Truth: {ground_truth} | Correct: {bool(correct)}")

    # Step 7: Logging
    result = {
        "input": input_prompt,
        "x0": x0,
        "p_expert": p_exp,
        "p_student": p_stu,
        "similarity": score,
        "x1": x1,
        "pred": pred,
        "label": ground_truth,
        "correct": bool(correct)
    }

    results.append(result)

    save_results_to_csv(
        result=result,
        features=["input", "x0", "p_expert", "p_student", "similarity", "x1", "pred", "label", "correct"],
        file_path="gsm8k_unified_results.csv"
    )

**Utility Functions**